# Vector databases

In this tutorial, we will explore vector databases in greater detail. Previously, we focused on smaller examples at the sentence level. Now, we will examine longer texts. For this tutorial, we will utilize [Wikipedia](https://www.wikipedia.org/) as a resource. The [Wikipedia library](https://pypi.org/project/wikipedia/) allows easy access to and utilization of Wikipedia articles, making it an excellent choice for our purposes. 

Our primary focus will be implementing strategies for dividing the text as well as calculating and storing embeddings in the vector database. Thereafter, we will query the vector database.


## Prerequisites

In this tutorial, we re-use the PostgreSQL database Docker image we have presented in the *introduction_to_vector_databases* notebook. If you have already tested the showcase example and have not deleted the container *postgres-wier* with the database, you can start the Docker container as indicated below.



<img src="https://i.ibb.co/DgCxJWdQ/docker-demo.png" alt="docker-demo" border="0">



Otherwise, follow next steps. 

First, prepare a file *database.sql*. The script will create a table with two rows:

``` sql
CREATE SCHEMA IF NOT EXISTS showcase;

CREATE TABLE showcase.counters (
    counter_id integer  NOT NULL,
    value integer NOT NULL,
    CONSTRAINT pk_counters PRIMARY KEY ( counter_id )
 );

INSERT INTO showcase.counters VALUES (1,0), (2,0);
```

Go to an empty folder and save the script into a subfolder named *init-scripts*. Create another empty folder named *pgdata*.

We run the docker container using the following command. The command will name the container *postgresql-wier*, set username and password, map database files to folder *./pgdata* and initialization scripts to *./init-scripts*, map port 5432 to host machine (i.e. localhost) and run image *pgvector:pg17* in a detached mode. 

``` 
docker run --name postgresql-wier \
    -e POSTGRES_PASSWORD=SecretPassword \
    -e POSTGRES_USER=user \
    -e POSTGRES_DB=wier \
    -v $PWD/pgdata:/var/lib/postgresql/data \
    -v $PWD/init-scripts:/docker-entrypoint-initdb.d \
    -p 5432:5432 \
    -d pgvector/pgvector:pg17
```

If you use Command Prompt on Windows, the equivalent of the above command is as follows:

``` 
docker run --name postgresql-wier ^
    -e POSTGRES_PASSWORD=SecretPassword ^
    -e POSTGRES_USER=user ^
    -e POSTGRES_DB=wier ^
    -v "%CD%\pgdata:/var/lib/postgresql/data" ^
    -v "%CD%\init-scripts:/docker-entrypoint-initdb.d" ^
    -p 5432:5432 ^
    -d pgvector/pgvector:pg17
```

To check the container's logs, run `docker logs -f postgresql-wier`.


## Getting started

First, install the necessary dependencies.

In [22]:
%pip install pgvector
%pip install sentence_transformers
%pip install numpy
%pip install nltk
%pip install wikipedia
%pip install beautifulsoup4
%pip install stanza

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


If you have already enabled the pgvector extension in the PostgreSQL database, you do not have to enable the pgvector extension again.  

In [23]:
from pgvector.psycopg import register_vector
import psycopg

#connect to db
conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')

#enable `vector` extension if not already enabled
conn.execute('CREATE EXTENSION IF NOT EXISTS vector')
register_vector(conn)
conn.close()

## Retrieval of articles from Wikipedia

We are now ready to begin our work. First, if the tables we need for this tutorial already exist, we will delete them. Then, we create the table showcase.wiki_articles with columns:
- *id*: primary key
- *topic*: topic of the Wikipedia page
- *content*: content of the Wikipedia page for the topic 

In [24]:
#connect to the db
conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')

#delete tables from the db if they already exist
conn.execute('DROP TABLE IF EXISTS showcase.wiki_chunks_sentences')
conn.execute('DROP TABLE IF EXISTS showcase.wiki_chunks_fixed_length')
conn.execute('DROP TABLE IF EXISTS showcase.wiki_articles')

#create table wiki_articles with columns id, topic and content
conn.execute('CREATE TABLE showcase.wiki_articles (id bigserial PRIMARY KEY, topic text, content text)')

conn.close()

In the first step, we get some Wikipedia articles by means of the Wikipedia library for Python for the selected topics. For demonstration purposes, we use the Slovenian language. Replace "sl" with "en" in the setup to obtain articles in English.

In [25]:
import wikipediaapi

# 1. Wikipedia API setup
USER_AGENT = "MySuperIEPSWikiBot/1.0" #define user agent
wiki_wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, #user agent
                                   language="en", #define the language. Replace "en" with "sl" to obtain articles in Slovenian.
                                   extract_format=wikipediaapi.ExtractFormat.WIKI)#to get the the HTML content use .HTML instead of .WIKI 

# 2. Define topics of the articles for retrieval
topics = ["Albert Einstein", "Taylor Swift", "Harry Potter", "J. K. Rowling", 
          "Magic Johnson", "Luka Dončić", "Peter Prevc", 
          "London", "Berlin", "Ljubljana",  
          "Bitcoin", "Jupiter", "Twitter" 
          ]

# 3. Fetch Wikipedia articles and store them in the db
conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
cur = conn.cursor() 
for topic in topics:
    print('Fetching content of the Wikipedia article for the topic: ' + topic)
    page = wiki_wiki.page(topic)
    if page.exists():
        print("URL of the page: " + page.fullurl)
        text = page.text # Get full content
        cur.execute('INSERT INTO showcase.wiki_articles (topic, content) VALUES (%s, %s)', (topic, text))
    else:
        print("No page found for the topic " + topic)
cur.close()
conn.close() 

Fetching content of the Wikipedia article for the topic: Albert Einstein
URL of the page: https://en.wikipedia.org/wiki/Albert_Einstein
Fetching content of the Wikipedia article for the topic: Taylor Swift
URL of the page: https://en.wikipedia.org/wiki/Taylor_Swift
Fetching content of the Wikipedia article for the topic: Harry Potter
URL of the page: https://en.wikipedia.org/wiki/Harry_Potter
Fetching content of the Wikipedia article for the topic: J. K. Rowling
URL of the page: https://en.wikipedia.org/wiki/J._K._Rowling
Fetching content of the Wikipedia article for the topic: Magic Johnson
URL of the page: https://en.wikipedia.org/wiki/Magic_Johnson
Fetching content of the Wikipedia article for the topic: Luka Dončić
URL of the page: https://en.wikipedia.org/wiki/Luka_Don%C4%8Di%C4%87
Fetching content of the Wikipedia article for the topic: Peter Prevc
URL of the page: https://en.wikipedia.org/wiki/Peter_Prevc
Fetching content of the Wikipedia article for the topic: London
URL of the

## Dividing the content, calculating and storing embeddings in vector database

In the next step, you have to extract the relevant content from the web page. It is of utmost importance to consider which content is relevant to your selected domain when doing Programming Assignment 2 (PA2). Also of note is that you will likely deal with lengthy text, which you will have to clean and divide into smaller portions, as most models for calculating embeddings have limitations on the number of words they can process at once.

As we have already downloaded the plain text of the articles, we skip this step of extracting relevant content and focus on dividing the text. There are [several approaches](https://www.analyticsvidhya.com/blog/2024/10/chunking-techniques-to-build-exceptional-rag-systems/) on how to divide text, e.g.:

1. Divide text into fixed-size segments.
2. Vary the sizes of text segments when dividing the text.
3. Sliding window.
4. ...

For the PA2, you will have to select the most appropriate strategy for dividing text and list its advantages and disadvantages. It is often necessary to conduct some testing to determine what works best. Without a proper approach, we risk overlooking important information or providing incomplete or out-of-context retrieved segments. Some of the key factors to consider when dividing the text into smaller portions are the size of segments and context preservation.

In this tutorial, we test two strategies. The first strategy divides the text into fixed-length segments, whereas the second strategy divides the text into text segments comprising sentences. The obtained text segment does not exceed the specified word count limit. We use the [nltk library](https://www.nltk.org/index.html).

In [26]:
import re
import nltk
import textwrap
from nltk.tokenize import sent_tokenize

# Download NLTK sentence tokenizer if not already installed
nltk.download("punkt")
nltk.download("punkt_tab")

def chunk_fixed_length(text, chunk_size=50):
    """Fixed length chunking."""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


def chunk_segments(text, max_words=256):
    """Splits text into sentence-based chunks with a max word count limit."""
    sentences = sent_tokenize(text)  # Split into sentences
    chunks, current_chunk = [], []
    current_length = 0
    
    for sentence in sentences:
        words = sentence.split()
        if current_length + len(words) > max_words:
            chunks.append(" ".join(current_chunk))  # Save current chunk
            current_chunk, current_length = [], 0  # Reset chunk
        current_chunk.append(sentence)
        current_length += len(words)
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))  # Add last chunk

    return chunks

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\melanija.vezocnik\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\melanija.vezocnik\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt_tab is already up-to-date!


Below are two functions we use in this tutorial. The first one obtaines Wikipedia articles from the database, whereas the second function creates two tables (showcase.wiki_chunks_fixed_length, showcase.wiki_chunks_sentences) for storing embeddings using different strategies. Both tables have columns as follows:
- *chunk_id*: primary key
- *chunk*: text segment
- *embeddings*: embedding for the text segment
- *fk_pageid*: id of the page that includes the text segment and foreign key

In [27]:
def get_wiki_articles():
    """Get Wikipedia articles from the table showcase.wiki_articles."""

    wiki_articles = [] 

    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor()
    cur.execute("SELECT id, topic, content FROM showcase.wiki_articles")
    for id, topic, content in cur.fetchall():
        wiki_articles.append((id, topic, content))
    cur.close()
    conn.close()

    return wiki_articles


def create_tables():
    """Create tables showcase.wiki_chunks_fixed_length and showcase.wiki_chunks_sentences."""
    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    
    conn.execute('DROP TABLE IF EXISTS showcase.wiki_chunks_fixed_length')
    conn.execute('DROP TABLE IF EXISTS showcase.wiki_chunks_sentences')
    
    conn.execute('CREATE TABLE showcase.wiki_chunks_fixed_length (chunk_id bigserial PRIMARY KEY, chunk text, embedding vector(768), fk_pageid bigserial)')
    conn.execute('CREATE TABLE showcase.wiki_chunks_sentences (chunk_id bigserial PRIMARY KEY, chunk text, embedding vector(768), fk_pageid bigserial)')
    conn.execute('ALTER TABLE showcase.wiki_chunks_fixed_length ADD CONSTRAINT fk_pageid FOREIGN KEY ( fk_pageid ) REFERENCES showcase.wiki_articles( id ) ON DELETE RESTRICT;')
    conn.execute('ALTER TABLE showcase.wiki_chunks_sentences ADD CONSTRAINT fk_pageid FOREIGN KEY ( fk_pageid ) REFERENCES showcase.wiki_articles( id ) ON DELETE RESTRICT;')

    conn.close()

Next, we calculate embeddings using the two strategies we defined above and store them in the vector database. We use the embedding model LaBSE of the Sentence Transformer library to calculate embeddings. One page worth exploring for the model selection is the [Hugging Face web page](https://huggingface.co/models).

In [28]:
from sentence_transformers import SentenceTransformer

wiki_articles = get_wiki_articles()
create_tables()
model = SentenceTransformer('sentence-transformers/LaBSE')

conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
cur = conn.cursor() 

for article in wiki_articles:
    id = article[0]
    topic = article[1]
    wiki_content = article[2]

    #segment wiki content
    chunks1 = chunk_fixed_length(wiki_content)
    chunks2 = chunk_segments(wiki_content)

    #calculate embeddings for each chunk and store them into db
    for single_chunk in chunks1:
        #print(single_chunk)
        embedding = model.encode(single_chunk).tolist()
        cur.execute('INSERT INTO showcase.wiki_chunks_fixed_length (chunk, embedding, fk_pageid) VALUES (%s, %s, %s)', (single_chunk, embedding, id))

    for single_chunk in chunks2:
        embedding = model.encode(single_chunk).tolist()
        cur.execute('INSERT INTO showcase.wiki_chunks_sentences (chunk, embedding, fk_pageid) VALUES (%s, %s, %s)', (single_chunk, embedding, id))

cur.close()
conn.close()

## Querying 

By default, *pgvector* performs an exact search when querying the database. Adding [index](https://github.com/pgvector/pgvector?tab=readme-ov-file#indexing) enables us to use approximate nearest neighbour (ANN) search. The *pgvector* supports two indexes, HSNW and IVFFlat. Compared to the IVFFlat, it achieves better performance. For this reason, it has been selected for this tutorial. When defining the HSNW index, you must add an index for each distance function you want to use for querying. 

In [29]:
conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
conn.execute('CREATE INDEX ON showcase.wiki_chunks_fixed_length USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64);')
conn.execute('CREATE INDEX ON showcase.wiki_chunks_sentences USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64);')
conn.close()

Below is the function *query_db_cosine* we use to retrieve the top 5 most similar sentences from a pgvector database based on the cosine distance.

In [30]:
#query using cosine distance
def query_db_cosine(query, model_name, table_name):
    """
    The query_db_cosine function retrieves the top 5 most similar sentences from a pgvector database based on cosine distance. 
    It uses a pre-trained SentenceTransformer model to encode the input query and then searches for the closest embeddings stored in the database.

    Parameters
    - query (str): The input text query to be searched.
    - model_name (str): The name of the SentenceTransformer model to be used for encoding the query.
    - table_name (str): The name of the table containing the stored sentence embeddings. Possible options are showcase.vector_demo and showcase.vector_demo2
    """
    
    #download the model (if you have already downloaded the model, comment the line below since the model will be loaded from the cache)
    #model = SentenceTransformer(model_name)

    #calculate embedding for the query
    query_embedding = model.encode(query).tolist()  

    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 

    # execute the query to fetch the top 5 most similar sentences based on cosine distance
    result = cur.execute(
        'SELECT chunk, 1 - (embedding <=> %s::vector) AS similarity '
        'FROM ' + table_name + ' ORDER BY similarity DESC LIMIT 5',
        (query_embedding,)  # pass the embedding twice, once for ordering and once for calculation
    ).fetchall()
    cur.close()
    conn.close()
    return result

Next, we will try some examples.

### Example 1

In [31]:
query = 'Who is Harry Potter?'  #Kdo je Harry Potter?
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_fixed_length'

query_db_cosine(query, model_name, table_name)

[(' The Wizarding World of Harry Potter, a Harry Pott', 0.6801130175590515),
 ('introduces Harry Potter. Harry is a wizard who liv', 0.6775883436203003),
 ('rry Potter Edition, Scene It? Harry Potter and Leg', 0.6435109376907349),
 ("A parent's guide to Harry Potter. InterVarsity Pre", 0.6209203230786142),
 ('\nThe Harry Potter series has been recognised by a ', 0.6152039399621534)]

In [ ]:
query = 'Who is Harry Potter?'  #Kdo je Harry Potter?
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_sentences'

query_db_cosine(query, model_name, table_name)

[("Harry Potter is a series of seven fantasy novels written by British author J. K. Rowling. The novels chronicle the lives of a young wizard, Harry Potter, and his friends, Ron Weasley and Hermione Granger, all of whom are students at Hogwarts School of Witchcraft and Wizardry. The main story arc concerns Harry's conflict with Lord Voldemort, a dark wizard who intends to become immortal, overthrow the wizard governing body known as the Ministry of Magic, and subjugate all wizards and non-magical people, known in-universe as Muggles. The series was originally published in English by Bloomsbury in the United Kingdom and Scholastic Press in the United States. A series of many genres, including fantasy, drama, coming-of-age fiction, and the British school story (which includes elements of mystery, thriller, adventure, horror, and romance), the world of Harry Potter explores numerous themes and includes many cultural meanings and references. Major themes in the series include prejudice, co

### Example 2

In [33]:
query = 'the capital of Germany' #glavno mesto Nemčije
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_fixed_length'
query_db_cosine(query, model_name, table_name)

[('Berlin is the capital of Germany, as well as its l', 0.6040120720863342),
 ('the major cities of Germany. the regional rail lin', 0.6009610891342163),
 (' is among the cities in Germany that received the ', 0.5858575105667114),
 ("s Germany's second-largest metropolitan region aft", 0.5850820541381836),
 ('e main railway hub and economic center of Germany.', 0.5780694484710693)]

In [34]:
query = 'the capital of Germany' #glavno mesto Nemčije
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_sentences'
query_db_cosine(query, model_name, table_name)

[("Berlin is the capital of Germany, as well as its largest city by both area and population. With 3.7 million inhabitants, it has the highest population within its city limits of any city in the European Union. The city is also one of the states of Germany, being the third-smallest state in the country by area. Berlin is surrounded by the state of Brandenburg, bordering Brandenburg's capital Potsdam to the southwest. The urban area of Berlin has a population of over 4.6 million, making it the most populous in Germany. The Berlin-Brandenburg capital region has around 6.2 million inhabitants and is Germany's second-largest metropolitan region after the Rhine-Ruhr region, as well as the fifth-biggest metropolitan region by GDP in the European Union. Berlin was built along the banks of the Spree river, which flows into the Havel in the western borough of Spandau. The city includes lakes in the western and southeastern boroughs, the largest of which is Müggelsee. About one-third of the cit

### Example 3

In [35]:
query = 'pop artist' #pop izvajalka
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_fixed_length'
query_db_cosine(query, model_name, table_name)

[('nger-songwriter to a pop star in the 2000s and 201', 0.5034947545377839),
 ('s and 1980s, such as pop rock, pop-punk, and arena', 0.47012221457031034),
 (' as Artist of the Decade by the American Music Awa', 0.463977052687915),
 (' American singer-songwriter. An influential figure', 0.4431160058448318),
 ('ift is the highest-grossing live music artist, the', 0.4402379262376468)]

In [36]:
query = 'pop artist' #pop izvajalka
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_sentences'
query_db_cosine(query, model_name, table_name)

[('She is the solo artist with the most weeks at number one on the Billboard 200; the female artist with the most number-one albums on the Billboard 200 (fifteen) and most number-one debuts on the Billboard Hot 100 (eight); the artist with the most number-one songs on Pop Airplay (fifteen) and Adult Pop Airplay (fifteen, tied with Maroon 5); the first living artist to chart five albums in the top 10 of the Billboard 200; the first woman to have both an album (Fearless) and a song ("Shake It Off") receive Diamond certifications from the Recording Industry Association of America (RIAA); and the first and only woman to have 100 million album units certified by the RIAA. Billboard ranked her at number eight on its list "Greatest of All Time Artists" (2019), number two on "Greatest Pop Stars of the 21st Century" (2024), and number one on "Top 100 Women Artists of the 21st Century" and "Top Artists of the 21st Century" (both 2025). Swift has appeared in power listings. In 2024, she became th

### Example 4

In [37]:
query = 'Taylor Swift' 
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_fixed_length'
query_db_cosine(query, model_name, table_name)

[(' Swift\nTaylor Swift at AllMusic \nTaylor Swift disc', 0.7980667590518824),
 ('ntry singer with the albums Taylor Swift (2006) an', 0.7168257877372912),
 ('he promotional film Taylor Swift: The Official Rel', 0.7051032009782315),
 ('nstaged: Taylor Swift Experience, for which she wo', 0.6829915439191028),
 ('istmas extended play (EP) The Taylor Swift Holiday', 0.6694445410536393)]

In [38]:
query = 'Taylor Swift' 
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_sentences'
query_db_cosine(query, model_name, table_name)

[('The contract was finalized by July 2005, when Swift ended the working relationship with Dymtrow. She spent four months near the end of 2005 recording her debut album, Taylor Swift, with the producer Nathan Chapman. Swift\'s debut single, "Tim McGraw", was released in June 2006. She and her mother spent mid-2006 sending promotional copies of the song to country radio stations across the US. Taylor Swift was released on October 24, 2006. On the US Billboard 200 chart, the album peaked at number five and spent 157 weeks on the chart—the longest chart run by an album in the 2000s. With Taylor Swift, she became the first female country music artist to write or co-write every track on a platinum-certified debut album. The album was promoted by a six-month radio tour and by Swift opening for other country artists, including Rascal Flatts in 2006, and George Strait, Brad Paisley, and Tim McGraw and Faith Hill in 2007. She opened for Rascal Flatts again in 2008, when she dated the singer Joe

### Example 5

In [39]:
query = 'ski jumper' #smučarski skakalec
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_fixed_length'
query_db_cosine(query, model_name, table_name)

[('ics\nFIS Nordic World Ski Championships\nFIS Ski Fly', 0.4651638865470886),
 ('icipated at the FIS Nordic World Ski Championships', 0.4253961443901062),
 (' ski jumping hill, built in 1954 to plans by Stank', 0.41979676485061646),
 ('h the Slovenia national team at the Ski Flying Wor', 0.39610105752944946),
 ('te \nPeter Prevc at FIS (ski jumping)\nPeter Prevc a', 0.39609054350313355)]

In [40]:
query = 'ski jumper' #smučarski skakalec
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_sentences'
query_db_cosine(query, model_name, table_name)

[('Peter Prevc (Slovene: [ˈpéːtəɾ ˈpɾéːwts]; born 20 September 1992) is a Slovenian former ski jumper. He won the 2016 Ski Jumping World Cup overall title and four Olympic medals, including gold at the 2022 Winter Olympics in the mixed team event. He also won the 2016 Four Hills Tournament, 2016 Ski Flying World Championships, and three consecutive Ski Flying World Cup overall titles (2014, 2015, and 2016). In addition, Prevc won two team events with the Slovenia national team at the Ski Flying World Championships, in 2022 and 2024. A specialist in ski flying, Prevc is a former world record holder and the first athlete in history to land a jump of 250 metres (820 ft). In 2015, in Planica, Prevc became one of the few ski jumpers in history to achieve a "perfect jump", with all five judges awarding him the maximum style points of 20. In the following year, Prevc achieved the most individual World Cup competition wins in a single season – 15 – which is also a record. Prevc was named Slove

### Example 6

In [41]:
query = 'potpourri' 
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_fixed_length'
query_db_cosine(query, model_name, table_name)

[(' Potter. In the first book, Harry Potter and the P', 0.39936720725102537),
 (' by Bloomsbury, the publisher of all Harry Potter ', 0.3956385913110626),
 ('ovels: the Harry Potter books are predominantly se', 0.39420144488829),
 (')\nHarry Potter at Bloomsbury (UK publisher)\nArchiv', 0.39235883951187134),
 (' The Wizarding World of Harry Potter, a Harry Pott', 0.3888753056526184)]

In [42]:
query = 'potpourri'
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.wiki_chunks_sentences'
query_db_cosine(query, model_name, table_name)

[('Nancy Stouffer sued Rowling in 1999, alleging that Harry Potter was based on stories she published in 1984. Rowling won in September 2002. Richard Posner describes Stouffer\'s suit as deeply flawed and notes that the court, finding Stouffer had used "forged and altered documents", assessed a $50,000 penalty against her. With her literary agents and Warner Bros., Rowling has brought legal action against publishers and writers of Harry Potter knockoffs in several countries. In mid-2000, Rowling and her publishers obtained a series of injunctions prohibiting sales or published reviews of her books before their official release dates. Beginning in 2001, after Rowling sold film rights to Warner Bros., the studio tried to take Harry Potter fan sites offline unless it determined that they were made by "authentic" fans for innocuous purposes. In 2007, with Warner Bros., Rowling started proceedings to cease publication of a book based on content from a fan site called The Harry Potter Lexico

### How to improve results?

When building a search system, retrieval alone often is not sufficient. You can obtain close results, but not truly the best match. For this reason, the reranking can be used. It adds a second layer of intelligence by comparing the query directly to each candidate, allowing the system to refine its choices and push the most meaningful results to the top. The [Sentence transformers](https://sbert.net/) library makes this easy, providing CrossEncoder models that specialize in evaluating query‑passage pairs with far greater precision than raw cosine similarity. By combining retrieval with a reranking step, you can obtain search results that are relevant to your query.

In [ ]:

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")


def rerank_candidates(query, candidates):
    """
    query: string
    candidates: list of (passage_text, cosine_score) from LaBSE retrieval

    returns: list of dicts sorted by cross-encoder score:
      [
         {
            "text": passage_text,
            "cosine": cosine_score,
            "cross_score": float
         },
         ...
      ]
    """
    # Extract only the texts in order
    passages = [c[0] for c in candidates]

    # Prepare pairs: [query, passage_text]
    cross_inputs = [[query, p] for p in passages]

    # Predict with the crossencoder
    cross_scores = reranker.predict(cross_inputs)

    print(cross_scores)

    # Merge everything together
    enriched = []
    for (text, cosine), cross in zip(candidates, cross_scores):
        enriched.append({
            "text": text,
            "cosine": float(cosine),
            "cross_score": float(cross)
        })

    # Sort by cross-score descending
    enriched.sort(key=lambda x: x["cross_score"], reverse=True)

    return enriched



# Example usage:
query = 'Who is Harry Potter?' 
table_name = 'showcase.wiki_chunks_sentences'

results = query_db_cosine(query, model_name, table_name)
ranked = rerank_candidates(query, results)

for item in ranked:
    print(f"{item['cross_score']:.4f} | cosine={item['cosine']:.4f}\n{item['text'][:120]}...\n")


[0.90240675 0.41056418 0.5023378  0.38914678 0.36404097]
0.9024 | cosine=0.4926
Harry Potter is a series of seven fantasy novels written by British author J. K. Rowling. The novels chronicle the lives...

0.5023 | cosine=0.4874
In the first book, Harry Potter and the Philosopher's Stone (Harry Potter and the Sorcerer's Stone in the US), Harry liv...

0.4106 | cosine=0.4878
Duriez, Colin (2007). Field Guide to Harry Potter. IVP Books. ISBN 978-0-8308-3430-3. Mulholland, Neil (2007). The Psych...

0.3891 | cosine=0.4617
This led J. K. Rowling to issue several statements urging Harry Potter fans to refrain from purchasing pet owls. Despite...

0.3640 | cosine=0.4595
Reception
Commercial success
The popularity of the Harry Potter series has translated into substantial financial success...



In [62]:
reranker = CrossEncoder("Alibaba-NLP/gte-multilingual-reranker-base",  trust_remote_code=True)

query = 'Kdo je Taylor Swift?' 
table_name = 'showcase.wiki_chunks_sentences'

results = query_db_cosine(query, model_name, table_name)
ranked = rerank_candidates(query, results)

for item in ranked:
    print(f"{item['cross_score']:.4f} | cosine={item['cosine']:.4f}\n{item['text'][:120]}...\n")

[0.68016154 0.4485689  0.9647545  0.48789796 0.54522747]
0.9648 | cosine=0.4346
Taylor Alison Swift (born December 13, 1989) is an American singer-songwriter. An influential figure in popular culture,...

0.6802 | cosine=0.4521
Her Eras Tour (2023–2024) and its associated film, Taylor Swift: The Eras Tour (2023), are the highest-grossing concert ...

0.5452 | cosine=0.3900
The critic Kelefa Sanneh dubbed Swift the biggest country star since Garth Brooks, "and maybe since before him, too". He...

0.4879 | cosine=0.4102
Discography
Filmography
Tours
Fearless Tour (2009–2010)
Speak Now World Tour (2011–2012)
The Red Tour (2013–2014)
The 19...

0.4486 | cosine=0.4502
The contract was finalized by July 2005, when Swift ended the working relationship with Dymtrow. She spent four months n...

